<a href="https://colab.research.google.com/github/iarithik/Notes-Classiq/blob/main/Classiq_Notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install classiq

In [ ]:
import classiq
classiq.authenticate()

If a browser doesn't automatically open, please visit this URL from any trusted device to authenticate: https://auth.classiq.io/authorize?client_id=f6721qMOVoDAOVkzrv8YaWassRKSFX6Y&response_type=code&audience=https%3A%2F%2Fcadmium-be&redirect_uri=https%3A%2F%2Fauth.classiq.io%2Factivate%3Fuser_code%3DCRTQ-HGGT&scope=offline_access
Your user code: CRTQ-HGGT


NotImplementedError: 

---


In [ ]:
from classiq import *

In [ ]:
@qfunc
def foo(q: QBit) -> None:
    X(q)
    H(q)

@qfunc
def main(q: Output[QBit]) -> None:
    allocate(q)
    foo(q)

qmod = create_model(main)
qprog = synthesize(qmod)

show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3ANhNe85eqFtYT4gEBuRYthRVJn


In [ ]:
job = execute(qprog)
pc = job.get_sample_result().parsed_counts
print(pc)

[{'q': 0}: 1038, {'q': 1}: 1010]


## Qmod Tutorial - Part 1

### Exercise #0

Rewrite the above model, so that q is initilized inside foo. A solution is provided in the end of the notebook. Hint: it only requires to move one line of code and add the Output modifier in the correct place.

In [ ]:
@qfunc
def foo(q: Output[QBit]) -> None:
    allocate(1, q)
    X(q)
    H(q)

@qfunc
def main(q: Output[QBit]) -> None:
    foo(q)

qmod = create_model(main)
qprog = synthesize(qmod)

show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3ANib3zgXBmX3RegFGUHVqYxY9K


### Exercise #1 - Quantum Arrays

After we have familiarized with the QBit varible type (which is simply a single qubit), it is a good timing to introduce the quantum array type QArray.

In this exercise, we will prepare the famous |phi_+> Bell state into a 2-qubit Quantum array.

Recall that |phi_+> represents the state 1/sqrt2 (|0> + |1>).

Instructions:

Declare a quantum variable qarr of type QArray, and initilize it by allocating to it 2 qubits. Don't forget to use the Output modifier.

Apply a Hadamard gate on the first qubit of qarr. Qmod counts from 0, so the first entry of qarr is qarr[0].

Apply CX (controlled-NOT gate), with the control parameter being qarr[0] and the target parameter being qarr[1].

Synthesize and execute your model to assure that
 and
 are the only states to be measured, and that they are measured roughly equally.

In [ ]:
@qfunc
def Bell(qarr: QArray[QBit, 2]) -> None:
    H(qarr[0])
    CX(qarr[0], qarr[1])

@qfunc
def main(qarr: Output[QArray]) -> None:
    allocate(2, qarr)
    Bell(qarr)

qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3ANkCwVO14bocZNi6ELNBEOMDoM


### Exercise #2 - The Repeat Statement

Use Qmod's repeat statement to create your own Hadamard Transform - a function that takes a QArray of an unspecified size and applies H to each of its qubits.

Instructions:

Define a function my_hadamard_transform:

It should have a single QArray argument q.

Use repeat to apply H on each of q's qubits.

Note that the iteration block of the repeat statement must use the Python lambda syntax (see repeat documentation).

define a main function that initializes a QArray of length 10, and then passes it to my_hadamard_transform.

The provided code continues by calling show to let you inspect the resulting circuit - make sure that is applies H to each of q's qubits.

In [ ]:
@qfunc
def hadamard_transform(q: QArray[QBit]) -> None:
    repeat(q.len, lambda i: H(q[i]))

@qfunc
def main(q: Output[QArray[QBit]]) -> None:
    allocate(10, q)
    hadamard_transform(q)

qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3ANmYU5wqJPSaErWDx7iDOtXEG7


### Exercise #3 - Power

Raising a quantum operation to an integer power appears in many known algorithms; for example, in Grover search and Quantum Phase Estimation. In the general case the implementation involves repeating the same circuit multiple times.

Sometimes, however, the implementation of the power operation can be simplified, thereby saving computational resources. A simple example is the operation of rotating a single qubit about the X, Y, or Z axis. In this case the rotation gate can be used once with the angle multiplied by the exponent. A similar example is the function unitary - an operation expressed as an explicit unitary matrix (i.e., all n*n matrix terms are given). Raising the operation can be done by raising the matrix to that power via classical computation.

See power operator.

Use the following code to define the value of a Qmod constant named unitary_matrix as a 4x4 (real) unitary:

Any Matrix 'A' can be written as QR decomposition, where Q is the unitary matrix and R is the upper triangular matrix.

In [ ]:
from typing import List

import numpy as np

from classiq import *

rng = np.random.default_rng(seed=0)
random_matrix = rng.random((4, 4))
qr_unitary, _ = np.linalg.qr(random_matrix)

unitary_matrix = QConstant("unitary_matrix", List[List[float]], qr_unitary.tolist())

Using power():

In [ ]:
@qfunc
def main(q: Output[QArray[QBit]]) -> None:
    allocate(2, q)
    power(3, lambda: unitary(unitary_matrix, q))

In [ ]:
qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3AODgL04hj84M9SupKE0iwDjVde


Using repeat():

In [ ]:
@qfunc
def main(q: Output[QArray[QBit]]) -> None:
    allocate(2, q)
    repeat(3, lambda i: unitary(unitary_matrix, q))

In [ ]:
qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3AOEnSfSmCDU2w6jEuxkSY2G1zY


### Exercise 4 - User-defined Operators

Create a function that applies a given single-qubit operation to all qubits in its quantum argument (call your function my_apply_to_all). Such a function is also called an operator; i.e., a function that takes another function as an argument (its operand).

See operators.

Follow these guidelines:

Your function declares a parameter of type qubit array and a parameter of a function type with a single qubit parameter.

The body applies the operand to all qubits in the argument.

Now, re-implement my_hadamard_transform from Exercise 2 using this function instead of repeat. Use the same main function from Exercise 2.

In [ ]:
@qfunc
def my_apply_to_all(operand: QCallable[QBit], q: QArray[QBit]) -> None:
    repeat(q.len, lambda i: operand(q[i]))

@qfunc
def my_hadamard_transform(q: QArray[QBit]) -> None:
    my_apply_to_all(lambda t: H(t), q)

@qfunc
def main(q: Output[QArray[QBit]]) -> None:
    allocate(10, q)
    hadamard_transform(q)

In [ ]:
qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3AOHqSWAOuVbtVO6XBKC04WtjqG


### Exercise 5 - Quantum Conditionals

#### Exercise 5a - Control Operator

Use the built-in control operator to create a function that receives two single qubit variables and uses one of them to control an RY gate with a pi/2 angle acting on the other variable (without using the CRY function).

See control.

In [ ]:
from classiq.qmod.symbolic import pi

@qfunc
def my_controlled_ry(control_bit: QBit, target: QBit) -> None:
    control(control_bit, lambda: RY(pi/2, target))

@qfunc
def main(control_bit: Output[QBit], target: Output[QBit]) -> None:
    allocate(1, control_bit)
    allocate(1, target)
    my_controlled_ry(control_bit, target)

In [ ]:
qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3AOJ1LQSSxwpa56a7jJbHwEe1aN


#### Exercise 5b - Control Operator with Quantum Expressions


The control operator is the conditional application of some operation, with the condition being that all control qubits are in the state |1>. This notion is generalized in Qmod to other control states, where the condition is specified as a comparison between a quantum numeric variable and a numeric value, similar to a classical if statement. Quantum numeric variables are declared with class QNum.

See numeric types.

Declare a QNum output argument using Output[QNum] and name it x.

Use numeric assignment (the |= operator) to initialize it to 9.

Execute the circuit and observe the results.

Declare another output argument of type QBit and perform a control such that if x is 9, the qubit is flipped. Execute the circuit and observe the results. Repeat for a different condition.

In [ ]:
@qfunc
def main(x: Output[QNum], target: Output[QBit]) -> None:
    x |= 9
    allocate(1, target)
    control(x == 9, lambda: X(target))

In [ ]:
qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3AOK4TLwfuptO10CLotrRq78bMe


## Walk-through: prepare_state

In [ ]:
probabilities = [
    0,
    0.002,
    0.004,
    0.006,
    0.0081,
    0.0101,
    0.0121,
    0.0141,
    0.0161,
    0.0181,
    0.0202,
    0.0222,
    0.0242,
    0.0262,
    0.0282,
    0.0302,
    0.0323,
    0.0343,
    0.0363,
    0.0383,
    0.0403,
    0.0423,
    0.0444,
    0.0464,
    0.0484,
    0.0504,
    0.0524,
    0.0544,
    0.0565,
    0.0585,
    0.0605,
    0.0625,
]


@qfunc
def main(io: Output[QArray]):
    prepare_state(probabilities=probabilities, bound=0.01, out=io)

In [ ]:
qprog = synthesize(main)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3AOL0Ax1UqVJS9FhkmO0WMNMQeM


In [ ]:
preferences = ExecutionPreferences(
    backend_preferences=ClassiqBackendPreferences(
        backend_name=ClassiqSimulatorBackendNames.SIMULATOR
    )
)

In [ ]:
with ExecutionSession(qprog, preferences) as es:
    res = es.sample()

In [ ]:
res.dataframe.head()

,io,counts,probability,bitstring
0,"[0, 0, 1, 1, 1]",133,0.064941,11100
1,"[1, 0, 1, 1, 1]",123,0.060059,11101
2,"[1, 1, 1, 1, 1]",121,0.059082,11111
3,"[1, 0, 0, 1, 1]",120,0.058594,11001
4,"[0, 1, 1, 1, 1]",117,0.057129,11110


## State Preparation

### Preparing Probabilities

In [ ]:
from classiq import *


@qfunc
def main(x: Output[QNum]):
    probabilities = [0.05, 0.11, 0.13, 0.23, 0.27, 0.12, 0.03, 0.06]
    prepare_state(probabilities=probabilities, bound=0.01, out=x)


qmod = create_model(main)

In [ ]:
qprog = synthesize(qmod)

Printing the probabilities

In [ ]:
import numpy as np

result = execute(qprog).result_value()

probs = np.zeros(8)
for sample in result.parsed_counts:
    probs[int(sample.state["x"])] = sample.shots / result.num_shots
print("Resulting probabilities:", probs)

Resulting probabilities: [0.04638672 0.10546875 0.14257812 0.24804688 0.24316406 0.12011719
 0.03759766 0.05664062]


So if you see, the probabilities changed (but still sum to 1) because we had applied a error bound which changed the probabilities a little bit (<= bound) so that the circuit complexity is reduced.

If you dont want it to change the probabilities then you can just say bound=0

### Preparing Amplitudes

In [ ]:
from classiq.execution import ClassiqBackendPreferences, ExecutionPreferences


@qfunc
def main(x: Output[QNum]):
    amps = np.linspace(-1, 1, 8)
    amps = amps / np.linalg.norm(amps)
    prepare_amplitudes(amplitudes=amps.tolist(), bound=0, out=x)


qmod = create_model(main)
qmod = set_execution_preferences(
    qmod,
    num_shots=1,
    backend_preferences=ClassiqBackendPreferences(backend_name="simulator_statevector"),
)

In [ ]:
qprog = synthesize(qmod)

Printing out the amplitudes

In [ ]:
import numpy as np

result = execute(qprog).result_value()

amps = np.zeros(8, dtype=complex)
for sample in result.parsed_state_vector:
    amps[int(sample.state["x"])] = sample.amplitude

# remove global phase
global_phase = np.angle(amps[0])
amps = np.real(amps / np.exp(1j * global_phase))

print("Resulting amplitudes:", amps)

Resulting amplitudes: [ 0.54006172  0.38575837  0.23145502  0.07715167 -0.07715167 -0.23145502
 -0.38575837 -0.54006172]
